In [4]:
!pip install -q tensorflow-datasets

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

TF version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Configuração

In [ ]:
IMG_SIZE    = 128
NUM_CLASSES = 3
BATCH_SIZE  = 32
EPOCHS      = 40
AUTOTUNE    = tf.data.AUTOTUNE

## 2. Dataset Oxford-IIIT Pet (completo) + augmentation

Normalização `pixel/255` (idêntica ao Android). Máscara {1,2,3} → {0,1,2}.
Augmentation só no treino: espelhamento horizontal (melhora a generalização para fotos suas).

In [ ]:
dataset, info = tfds.load('oxford_iiit_pet:4.*.*', with_info=True)

def normalize(image, mask):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method='bilinear')
    mask  = tf.image.resize(mask,  (IMG_SIZE, IMG_SIZE), method='nearest')
    image = tf.cast(image, tf.float32) / 255.0
    mask  = tf.cast(mask, tf.int32) - 1
    return image, mask

def load(d):
    return normalize(d['image'], d['segmentation_mask'])

def augment(image, mask):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask  = tf.image.flip_left_right(mask)
    return image, mask

n_train = info.splits['train'].num_examples
n_test  = info.splits['test'].num_examples
print('Treino:', n_train, '| Teste:', n_test)

train_ds = (dataset['train']
            .map(load, num_parallel_calls=AUTOTUNE)
            .cache()
            .shuffle(1000)
            .map(augment, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE).prefetch(AUTOTUNE))

test_ds  = (dataset['test']
            .map(load, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE).prefetch(AUTOTUNE))

### Espiando uma amostra

In [ ]:
for img, msk in train_ds.take(1):
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1); plt.title('Imagem');  plt.imshow(img[0]); plt.axis('off')
    plt.subplot(1, 2, 2); plt.title('Máscara'); plt.imshow(msk[0, ..., 0]); plt.axis('off')
    plt.show()

## 3. Modelo — U-Net com encoder MobileNetV2 (transfer learning)

O encoder usa pesos da ImageNet (congelados). O decoder (Conv2DTranspose) treina do zero.
Tudo usa operações que convertem bem para TFLite.

In [ ]:
from tensorflow.keras import layers


base = tf.keras.applications.MobileNetV2(
    input_shape=[IMG_SIZE, IMG_SIZE, 3], include_top=False, weights='imagenet')

skip_names = [
    'block_1_expand_relu',
    'block_3_expand_relu',
    'block_6_expand_relu',
    'block_13_expand_relu',
    'block_16_project',
]
base_outputs = [base.get_layer(n).output for n in skip_names]
down_stack = tf.keras.Model(inputs=base.input, outputs=base_outputs, name='mobilenet_encoder')
down_stack.trainable = False

def upsample(filters, size=3):
    return tf.keras.Sequential([
        layers.Conv2DTranspose(filters, size, strides=2, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
    ])

up_stack = [upsample(512), upsample(256), upsample(128), upsample(64)]

def build_model(output_channels=NUM_CLASSES):
    inputs = layers.Input(shape=[IMG_SIZE, IMG_SIZE, 3])
    skips = down_stack(inputs)
    x = skips[-1]
    skips = reversed(skips[:-1])
    for up, skip in zip(up_stack, skips):
        x = up(x)
        x = layers.Concatenate()([x, skip])

    x = layers.Conv2DTranspose(output_channels, 3, strides=2, padding='same')(x)
    return tf.keras.Model(inputs=inputs, outputs=x, name='unet_mobilenet')

model = build_model()
model.summary()

## 4. Treino

Saída em logits → `from_logits=True`. `EarlyStopping` guarda o melhor modelo e evita treino à toa.

In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

early = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=6, restore_best_weights=True)

history = model.fit(train_ds, validation_data=test_ds, epochs=EPOCHS, callbacks=[early])

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.plot(history.history['loss'], label='treino')
plt.plot(history.history['val_loss'], label='val'); plt.title('Loss'); plt.legend()
plt.subplot(1, 2, 2); plt.plot(history.history['accuracy'], label='treino')
plt.plot(history.history['val_accuracy'], label='val'); plt.title('Acurácia'); plt.legend()
plt.show()

## 4b. (Opcional) Fine-tuning do encoder

Para o **melhor** resultado: descongela o MobileNetV2 e treina mais um pouco com learning rate
bem baixo. Pode pular esta célula se já estiver satisfeito com o resultado acima.

In [ ]:

down_stack.trainable = True
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)
early_ft = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=8, restore_best_weights=True, verbose=1)
model.fit(train_ds, validation_data=test_ds, epochs=25, callbacks=[early_ft])

## 5. Avaliação — Acurácia e IoU

IoU via `tf.keras.metrics.MeanIoU` (equivalente TF ao `torchmetrics`).

In [ ]:
loss, acc = model.evaluate(test_ds, verbose=0)
print(f'Acurácia (teste): {acc:.4f}')

miou = tf.keras.metrics.MeanIoU(num_classes=NUM_CLASSES)
for img, msk in test_ds:
    pred = np.argmax(model.predict(img, verbose=0), axis=-1)
    miou.update_state(msk[..., 0], pred)
print(f'Mean IoU (teste): {miou.result().numpy():.4f}')

## 6. Visualização — imagem + máscara sobreposta

In [3]:
PALETTE = np.array([[255, 0, 0], [0, 0, 0], [0, 0, 255]], dtype=np.uint8)
ALPHA   = np.array([0.5, 0.0, 0.5])

def overlay(image01, classmap):
    base = (image01 * 255).astype(np.uint8)
    color = PALETTE[classmap]
    a = ALPHA[classmap][..., None]
    return (base * (1 - a) + color * a).astype(np.uint8)

for img, msk in test_ds.take(1):
    preds = np.argmax(model.predict(img, verbose=0), axis=-1)
    n = min(3, img.shape[0])
    plt.figure(figsize=(10, 3 * n))
    for i in range(n):
        plt.subplot(n, 3, i*3 + 1); plt.title('Original');   plt.imshow(img[i]); plt.axis('off')
        plt.subplot(n, 3, i*3 + 2); plt.title('Predição');   plt.imshow(preds[i]); plt.axis('off')
        plt.subplot(n, 3, i*3 + 3); plt.title('Sobreposta'); plt.imshow(overlay(img[i].numpy(), preds[i])); plt.axis('off')
    plt.tight_layout(); plt.show()

NameError: name 'np' is not defined

## 7. Exportar para TensorFlow Lite (FP32)

Sem quantização → FP32. Gera `model.tflite`.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

import os
print(f'model.tflite — {os.path.getsize("model.tflite")/1e6:.2f} MB')

### Conferir entrada/saída

In [2]:
try:
    from ai_edge_litert.interpreter import Interpreter
    interp = Interpreter(model_path='model.tflite')
except Exception:
    interp = tf.lite.Interpreter(model_path='model.tflite')
interp.allocate_tensors()
print('Input :', interp.get_input_details()[0]['shape'],  interp.get_input_details()[0]['dtype'])
print('Output:', interp.get_output_details()[0]['shape'], interp.get_output_details()[0]['dtype'])

NameError: name 'tf' is not defined

### Baixar o modelo
Coloque o `model.tflite` em `android/app/src/main/assets/` e rode o app de novo.

In [1]:
try:
    from google.colab import files
    files.download('model.tflite')
except Exception as e:
    print('Fora do Colab? Baixe model.tflite manualmente.', e)

Fora do Colab? Baixe model.tflite manualmente. Cannot find file: model.tflite
